In [ ]:
import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.formatting.rule import FormulaRule

# Styling profiles
font_title = Font(name="Segoe UI", size=16, bold=True, color="FFFFFF")
font_section = Font(name="Segoe UI", size=13, bold=True, color="1B365D")
font_header = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
font_bold = Font(name="Calibri", size=11, bold=True)
font_data = Font(name="Calibri", size=11)

fill_blue_header = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid")
fill_gray_header = PatternFill(start_color="595959", end_color="595959", fill_type="solid")
fill_section = PatternFill(start_color="DCE6F1", end_color="DCE6F1", fill_type="solid")
fill_input_req = PatternFill(start_color="FFF2CC", end_color="FFF2CC", fill_type="solid")
fill_accent_curr = PatternFill(start_color="E2EFDA", end_color="E2EFDA", fill_type="solid")
fill_accent_prop = PatternFill(start_color="FCE4D6", end_color="FCE4D6", fill_type="solid")

alignment_center = Alignment(horizontal="center", vertical="center", wrap_text=True)
alignment_left = Alignment(horizontal="left", vertical="center")
alignment_right = Alignment(horizontal="right", vertical="center")

thin_side = Side(border_style="thin", color="D9D9D9")
border_thin = Border(left=thin_side, right=thin_side, top=thin_side, bottom=thin_side)

def create_v15_workbook():
    wb = openpyxl.Workbook()
    
    ws_dash = wb.active
    ws_dash.title = "1. Dashboard & Results"
    ws_current = wb.create_sheet(title="2. Data Entry (Current)")
    ws_proposed = wb.create_sheet(title="3. Data Entry (Proposed)")
    ws_engine = wb.create_sheet(title="4. Simulation Engine")

    for ws in [ws_dash, ws_current, ws_proposed, ws_engine]:
        ws.views.sheetView[0].showGridLines = True

    # ==============================================================================
    # TAB 1: DASHBOARD
    # ==============================================================================
    ws_dash.merge_cells("A1:E1")
    title_cell = ws_dash["A1"]
    title_cell.value = "OPEN FAIR\u2122 QUANTITATIVE RISK COMPLIANCE MATRIX"
    title_cell.font = font_title
    title_cell.fill = fill_blue_header
    title_cell.alignment = alignment_center
    ws_dash.row_dimensions[1].height = 40

    ws_dash.cell(row=3, column=1, value="RISK ASSESSMENT SUMMARY STRATA").font = font_section
    ws_dash.merge_cells("A3:E3")
    ws_dash["A3"].fill = fill_section

    dash_headers = ["Risk Metrics", "Current Posture", "Proposed Posture", "Delta Change", "Delta %"]
    for idx, h in enumerate(dash_headers, start=1):
        c = ws_dash.cell(row=4, column=idx, value=h)
        c.font = font_header
        c.fill = fill_blue_header
        c.alignment = alignment_center
    ws_dash.row_dimensions[4].height = 25

    metrics = [
        ("Maximum Loss Exposure", "MAX"),
        ("90th Percentile Exposure", "PERCENTILE", 0.9),
        ("Average / Mean Loss Exposure", "AVERAGE"),
        ("10th Percentile Exposure", "PERCENTILE", 0.1),
        ("Minimum Loss Exposure", "MIN"),
        ("Likelihood of Loss Events", "COUNTIF")
    ]

    for i, m_tuple in enumerate(metrics):
        r = 5 + i
        ws_dash.cell(row=r, column=1, value=m_tuple[0]).font = font_bold
        
        if m_tuple[1] == "COUNTIF":
            # FIX: use \" (single backslash escape) not \\" (double backslash) so Excel sees plain " characters
            ws_dash.cell(row=r, column=2, value=f"=COUNTIF('{ws_engine.title}'!$V$4:$V$1003,\">0\")/1000")
            ws_dash.cell(row=r, column=3, value=f"=COUNTIF('{ws_engine.title}'!$AQ$4:$AQ$1003,\">0\")/1000")
            ws_dash.cell(row=r, column=2).number_format = "0.0%"
            ws_dash.cell(row=r, column=3).number_format = "0.0%"
        else:
            p_arg = f", {m_tuple[2]}" if len(m_tuple) == 3 else ""
            ws_dash.cell(row=r, column=2, value=f"={m_tuple[1]}('{ws_engine.title}'!$V$4:$V$1003{p_arg})")
            ws_dash.cell(row=r, column=3, value=f"={m_tuple[1]}('{ws_engine.title}'!$AQ$4:$AQ$1003{p_arg})")
            ws_dash.cell(row=r, column=2).number_format = "$#,##0"
            ws_dash.cell(row=r, column=3).number_format = "$#,##0"
        
        ws_dash.cell(row=r, column=4, value=f"=C{r}-B{r}")
        ws_dash.cell(row=r, column=5, value=f"=IF(B{r}=0,0,(C{r}-B{r})/B{r})")
        ws_dash.cell(row=r, column=4).number_format = "$#,##0;($#,##0);\"-\""
        ws_dash.cell(row=r, column=5).number_format = "0.0%"
        
        for c in range(1, 6):
            cell = ws_dash.cell(row=r, column=c)
            cell.border = border_thin
            if c > 1:
                cell.alignment = alignment_right

    ws_dash.cell(row=13, column=1, value="CONTROL INVESTMENT EFFICIENCY VECTORS").font = font_section
    ws_dash.merge_cells("A13:C13")
    ws_dash["A13"].fill = fill_section

    ws_dash.cell(row=14, column=1, value="Proposed Security Investment CapEx/OpEx:").font = font_bold
    inv_cell = ws_dash.cell(row=14, column=2, value=0)
    inv_cell.font = font_bold
    inv_cell.fill = fill_input_req
    inv_cell.number_format = "$#,##0"
    inv_cell.alignment = alignment_right

    ws_dash.cell(row=15, column=1, value="Net Risk Mitigation Benefit:").font = font_bold
    ws_dash.cell(row=15, column=2, value="=B7-C7").number_format = "$#,##0"
    ws_dash.cell(row=15, column=2).font = font_bold

    ws_dash.cell(row=16, column=1, value="Return on Security Investment (ROSI):").font = font_bold
    ws_dash.cell(row=16, column=2, value="=IF(B14=0,0,(B15-B14)/B14)").number_format = "0.0%"
    ws_dash.cell(row=16, column=2).font = font_bold

    for r in range(14, 17):
        for col in range(1, 4):
            ws_dash.cell(row=r, column=col).border = border_thin

    # ==============================================================================
    # Default v9.0 Values Injection Mapping
    # ==============================================================================
    v9_curr_defaults = {
        8: (1, 12, 52, "HIGH"),
        9: (0.1, 0.5, 0.9, "MEDIUM"),
        10: (1, 6, 40, "MEDIUM"),
        11: (0.2, 0.5, 0.8, "MEDIUM"),
        12: (0.3, 0.6, 0.9, "HIGH"),
        13: (0.1, 0.35, 0.75, "HIGH"),
        14: (1, 4, 12, "MEDIUM"),
        18: (5000, 20000, 100000, "HIGH"), 
        19: (50000, 200000, 800000, "HIGH"), 
        20: (0, 0, 0, "LOW"), 
        23: (0.95, 1, 1, "HIGH"), 
        24: (50000, 200000, 800000, "HIGH"), 
        25: (0, 0.02, 0.1, "LOW"), 
        26: (100000, 500000, 3000000, "LOW"), 
        27: (0.01, 0.15, 0.4, "MEDIUM"), 
        28: (10000, 250000, 5000000, "MEDIUM"), 
        29: (0.05, 0.2, 0.5, "MEDIUM"), 
        30: (10000, 100000, 500000, "MEDIUM"), 
    }

    v9_prop_defaults = {
        8: (1, 12, 52, "HIGH"),
        9: (0.1, 0.4, 0.8, "MEDIUM"),
        10: (1, 6, 40, "MEDIUM"),
        11: (0.2, 0.5, 0.8, "MEDIUM"),
        12: (0.55, 0.8, 0.98, "HIGH"),
        13: (0.05, 0.15, 0.4, "HIGH"),
        14: (1, 2, 8, "MEDIUM"),
        18: (5000, 20000, 100000, "HIGH"), 
        19: (40000, 150000, 600000, "HIGH"), 
        20: (0, 0, 0, "LOW"), 
        23: (0.95, 1, 1, "HIGH"), 
        24: (40000, 150000, 600000, "HIGH"), 
        25: (0, 0.01, 0.05, "LOW"), 
        26: (100000, 400000, 2000000, "LOW"), 
        27: (0.01, 0.1, 0.3, "MEDIUM"), 
        28: (10000, 150000, 3000000, "MEDIUM"), 
        29: (0.02, 0.1, 0.35, "MEDIUM"), 
        30: (10000, 100000, 500000, "MEDIUM"), 
    }

    # ==============================================================================
    # TABS 2 & 3: DATA ENTRY SHEETS
    # ==============================================================================
    def populate_data_entry_sheet(ws, fill_accent, defaults):
        ws.merge_cells("A1:G1")
        t_cell = ws["A1"]
        t_cell.value = f"OPEN FAIR\u2122 RISK PARAMETERS - {ws.title.upper()}"
        t_cell.font = font_title
        t_cell.fill = fill_blue_header
        t_cell.alignment = alignment_center
        ws.row_dimensions[1].height = 40

        ws.cell(row=3, column=1, value="CORE SCENARIO SETUP MODEL").font = font_section
        ws.merge_cells("A3:G3")
        ws["A3"].fill = fill_section

        ws.cell(row=4, column=1, value="Taxonomy Input Level Selection:").font = font_bold
        ws.cell(row=4, column=3, value="5. CF + PoA + TCap + RS").fill = fill_input_req
        
        # FIX: use \" (single escape) so Excel sees plain " not backslash+quote
        ws.cell(row=4, column=5, value='=IF(LEFT(C4,1)="1",1,IF(LEFT(C4,1)="2",2,IF(LEFT(C4,1)="3",3,IF(LEFT(C4,1)="4",4,5))))')
        ws.column_dimensions['E'].hidden = True

        dv_route = DataValidation(type="list", formula1='"1. Direct LEF,2. TEF + Vulnerability,3. TEF + TCap + RS,4. CF + PoA + Vulnerability,5. CF + PoA + TCap + RS"', allow_blank=False)
        ws.add_data_validation(dv_route)
        dv_route.add(ws["C4"])

        headers = ["FAIR Taxonomy Parameter Nodes", "Minimum", "Most Likely", "Maximum", "Confidence Factor (PERT)", "Context / Logic Rationale", "Data Sources References"]
        for idx, h in enumerate(headers, start=1):
            c = ws.cell(row=6, column=idx, value=h)
            c.font = font_header
            c.fill = fill_blue_header
            c.alignment = alignment_center
        ws.row_dimensions[6].height = 25

        structure = [
            (7, "PRIMARY FREQUENCY STRATUM CONTROLS", True, None),
            (8, "Contact Frequency (CF) [Events per Year]", False, "0.0"),
            (9, "Probability of Action (PoA) [0% - 100%]", False, "0.0%"),
            (10, "Threat Event Frequency (TEF) [Calculated / Overridden]", False, "0.0"),
            (11, "Threat Capability (TCap) [Percentile 0% - 100%]", False, "0.0%"),
            (12, "Resistance Strength (RS) [Percentile 0% - 100%]", False, "0.0%"),
            (13, "Vulnerability (Vuln) [Calculated / Overridden]", False, "0.0%"),
            (14, "Loss Event Frequency (LEF) [Calculated / Overridden]", False, "0.0"),
            (16, "PRIMARY LOSS MAGNITUDE VECTORS", True, None),
            (18, "Productivity Loss (ProdL)", False, "$#,##0"),
            (19, "Response Cost (RespC)", False, "$#,##0"),
            (20, "Replacement Cost (ReplC)", False, "$#,##0"),
            (22, "SECONDARY LOSS MAGNITUDE VECTORS", True, None),
            (23, "Secondary Response: Event Frequency (%)", False, "0.0%"),
            (24, "Secondary Response: Loss Magnitude ($)", False, "$#,##0"),
            (25, "Competitive Advantage: Event Frequency (%)", False, "0.0%"),
            (26, "Competitive Advantage: Loss Magnitude ($)", False, "$#,##0"),
            (27, "Fines & Judgments: Event Frequency (%)", False, "0.0%"),
            (28, "Fines & Judgments: Loss Magnitude ($)", False, "$#,##0"),
            (29, "Reputation Damage: Event Frequency (%)", False, "0.0%"),
            (30, "Reputation Damage: Loss Magnitude ($)", False, "$#,##0"),
        ]

        dv_conf = DataValidation(type="list", formula1='"LOW,MEDIUM,HIGH"', allow_blank=False)
        ws.add_data_validation(dv_conf)

        for r_idx, label, is_sec, num_fmt in structure:
            if is_sec:
                ws.cell(row=r_idx, column=1, value=label).font = font_section
                ws.merge_cells(start_row=r_idx, start_column=1, end_row=r_idx, end_column=7)
                ws.cell(row=r_idx, column=1).fill = fill_accent
                continue
            
            # Injecting Default v9.0 Values
            d_min, d_ml, d_max, d_conf = defaults.get(r_idx, (0, 0, 0, "MEDIUM"))
            
            ws.cell(row=r_idx, column=1, value=label).font = font_bold
            ws.cell(row=r_idx, column=2, value=d_min).fill = fill_input_req
            ws.cell(row=r_idx, column=3, value=d_ml).fill = fill_input_req
            ws.cell(row=r_idx, column=4, value=d_max).fill = fill_input_req
            
            c_cell = ws.cell(row=r_idx, column=5, value=d_conf)
            c_cell.fill = fill_input_req
            c_cell.alignment = alignment_center
            dv_conf.add(c_cell)
            
            ws.cell(row=r_idx, column=6, value="[Context Documentation]")
            ws.cell(row=r_idx, column=7, value="[Evidence Telemetry]")
            
            for col in range(2, 5):
                ws.cell(row=r_idx, column=col).number_format = num_fmt
                ws.cell(row=r_idx, column=col).alignment = alignment_right
            
            for col in range(1, 8):
                ws.cell(row=r_idx, column=col).border = border_thin

        # --- DYNAMIC ACTIVATION/DEACTIVATION CONDITIONAL FORMATTING ---
        fill_disabled = PatternFill(start_color="F2F2F2", end_color="F2F2F2", fill_type="solid")
        font_disabled = Font(color="A6A6A6")

        # CF & PoA (Rows 8, 9) -> Inactive if Level < 4
        ws.conditional_formatting.add("A8:G9", FormulaRule(formula=['$E$4<4'], fill=fill_disabled, font=font_disabled))
        
        # TEF (Row 10) -> Inactive if Level is 1, 4, or 5
        ws.conditional_formatting.add("A10:G10", FormulaRule(formula=['OR($E$4=1, $E$4=4, $E$4=5)'], fill=fill_disabled, font=font_disabled))
        
        # TCap & RS (Rows 11, 12) -> Inactive if Level is 1, 2, or 4
        ws.conditional_formatting.add("A11:G12", FormulaRule(formula=['OR($E$4=1, $E$4=2, $E$4=4)'], fill=fill_disabled, font=font_disabled))
        
        # Vuln (Row 13) -> Inactive if Level is 1, 3, or 5
        ws.conditional_formatting.add("A13:G13", FormulaRule(formula=['OR($E$4=1, $E$4=3, $E$4=5)'], fill=fill_disabled, font=font_disabled))
        
        # LEF (Row 14) -> Inactive if Level > 1
        ws.conditional_formatting.add("A14:G14", FormulaRule(formula=['$E$4>1'], fill=fill_disabled, font=font_disabled))

    populate_data_entry_sheet(ws_current, fill_accent_curr, v9_curr_defaults)
    populate_data_entry_sheet(ws_proposed, fill_accent_prop, v9_prop_defaults)

    # ==============================================================================
    # TAB 4: SIMULATION ENGINE
    # ==============================================================================
    ws_engine.cell(row=1, column=58, value="METADATA DISTRIBUTION BOUNDS MATRIX").font = font_bold
    meta_headers = ["Factor Key", "C_Min", "C_Max", "C_Alpha", "C_Beta", "C_Mean", "P_Min", "P_Max", "P_Alpha", "P_Beta", "P_Mean", "C_Lambda", "P_Lambda"]
    for idx, h in enumerate(meta_headers, start=58):
        c = ws_engine.cell(row=2, column=idx, value=h)
        c.font = font_header
        c.fill = fill_gray_header

    c_map = {
        "CF": 8, "PoA": 9, "TEF": 10, "TCap": 11, "RS": 12, "Vuln": 13, "LEF": 14,
        "ProdL": 18, "RespC": 19, "ReplC": 20,
        "S_Resp_F": 23, "S_Resp_M": 24, "S_Comp_F": 25, "S_Comp_M": 26,
        "S_Fine_F": 27, "S_Fine_M": 28, "S_Repu_F": 29, "S_Repu_M": 30
    }
    
    for p_row in sorted(list(set(c_map.values()))):
        ws_engine.cell(row=p_row, column=58, value=f"PARAM_{p_row}")
        ws_engine.cell(row=p_row, column=59, value=f"='{ws_current.title}'!$B${p_row}")
        ws_engine.cell(row=p_row, column=60, value=f"='{ws_current.title}'!$D${p_row}")
        
        # FIX: use single-escaped \" so Excel sees plain " characters in the formula
        ws_engine.cell(row=p_row, column=69, value=f"=IF('{ws_current.title}'!$E${p_row}=\"LOW\",2,IF('{ws_current.title}'!$E${p_row}=\"HIGH\",8,4))")
        ws_engine.cell(row=p_row, column=70, value=f"=IF('{ws_proposed.title}'!$E${p_row}=\"LOW\",2,IF('{ws_proposed.title}'!$E${p_row}=\"HIGH\",8,4))")
        
        c_mean = f"(('{ws_current.title}'!$B${p_row}+BQ{p_row}*'{ws_current.title}'!$C${p_row}+'{ws_current.title}'!$D${p_row})/(BQ{p_row}+2))"
        ws_engine.cell(row=p_row, column=63, value=f"={c_mean}")
        
        ws_engine.cell(row=p_row, column=61, value=f"=IF(BH{p_row}=BG{p_row},1,(((BK{p_row}-BG{p_row})*(BQ{p_row}+2))/(BH{p_row}-BG{p_row})))")
        ws_engine.cell(row=p_row, column=62, value=f"=IF(BH{p_row}=BG{p_row},1,(BQ{p_row}+2-BI{p_row}))")
        
        ws_engine.cell(row=p_row, column=64, value=f"='{ws_proposed.title}'!$B${p_row}")
        ws_engine.cell(row=p_row, column=65, value=f"='{ws_proposed.title}'!$D${p_row}")
        p_mean = f"(('{ws_proposed.title}'!$B${p_row}+BR{p_row}*'{ws_proposed.title}'!$C${p_row}+'{ws_proposed.title}'!$D${p_row})/(BR{p_row}+2))"
        ws_engine.cell(row=p_row, column=68, value=f"={p_mean}")
        
        ws_engine.cell(row=p_row, column=66, value=f"=IF(BM{p_row}=BL{p_row},1,(((BP{p_row}-BL{p_row})*(BR{p_row}+2))/(BM{p_row}-BL{p_row})))")
        ws_engine.cell(row=p_row, column=67, value=f"=IF(BM{p_row}=BL{p_row},1,(BR{p_row}+2-BN{p_row}))")

    def get_stable_pert(p_row, is_proposed=False):
        if not is_proposed:
            return f"IF($BG${p_row}=$BH${p_row},$BG${p_row},$BG${p_row}+($BH${p_row}-$BG${p_row})*BETA.INV(RAND(),$BI${p_row},$BJ${p_row}))"
        else:
            return f"IF($BL${p_row}=$BM${p_row},$BL${p_row},$BL${p_row}+($BM${p_row}-$BL${p_row})*BETA.INV(RAND(),$BN${p_row},$BO${p_row}))"

    def get_sec_loss_formula(is_proposed, f_idx, m_idx):
        pert_f = get_stable_pert(f_idx, is_proposed)
        pert_m = get_stable_pert(m_idx, is_proposed)
        return f"IF(RAND()<MAX(0,MIN(1,{pert_f})),MAX(0,{pert_m}),0)"

    def get_prim_loss_formula(is_proposed):
        return f"=MAX(0,{get_stable_pert(c_map['ProdL'], is_proposed)})+MAX(0,{get_stable_pert(c_map['RespC'], is_proposed)})+MAX(0,{get_stable_pert(c_map['ReplC'], is_proposed)})"

    engine_headers = [
        "Iteration", 
        "Current Contact Frequency (CF)", "Current Probability of Action (PoA)", "Current Threat Event Frequency (TEF)", "Current Threat Capability (TCap)", "Current Resistance Strength (RS)", "Current Susceptibility / Vulnerability (Susc)", "Current Loss Event Frequency Raw (LEF Raw)", "Current Loss Event Count (Rounded LEF)",
        "Current Primary Event Loss 1 (L1)", "Current Primary Event Loss 2 (L2)", "Current Primary Event Loss 3 (L3)", "Current Primary Event Loss 4 (L4)", "Current Primary Event Loss 5 (L5)",
        "Current Secondary Event Loss 1 (SL1)", "Current Secondary Event Loss 2 (SL2)", "Current Secondary Event Loss 3 (SL3)", "Current Secondary Event Loss 4 (SL4)", "Current Secondary Event Loss 5 (SL5)",
        "Current Aggregated Primary Losses", "Current Aggregated Secondary Losses", "Current Total Simulated Annual Risk Exposure",
        "Proposed Contact Frequency (CF)", "Proposed Probability of Action (PoA)", "Proposed Threat Event Frequency (TEF)", "Proposed Threat Capability (TCap)", "Proposed Resistance Strength (RS)", "Proposed Susceptibility / Vulnerability (Susc)", "Proposed Loss Event Frequency Raw (LEF Raw)", "Proposed Loss Event Count (Rounded LEF)",
        "Proposed Primary Event Loss 1 (L1)", "Proposed Primary Event Loss 2 (L2)", "Proposed Primary Event Loss 3 (L3)", "Proposed Primary Event Loss 4 (L4)", "Proposed Primary Event Loss 5 (L5)",
        "Proposed Secondary Event Loss 1 (SL1)", "Proposed Secondary Event Loss 2 (SL2)", "Proposed Secondary Event Loss 3 (SL3)", "Proposed Secondary Event Loss 4 (SL4)", "Proposed Secondary Event Loss 5 (SL5)",
        "Proposed Aggregated Primary Losses", "Proposed Aggregated Secondary Losses", "Proposed Total Simulated Annual Risk Exposure"
    ]

    for idx, header in enumerate(engine_headers, start=1):
        cell = ws_engine.cell(row=3, column=idx, value=header)
        cell.font = font_header
        cell.fill = fill_blue_header
        cell.alignment = alignment_center
    ws_engine.row_dimensions[3].height = 28

    for i in range(1, 1001):
        r = i + 3
        ws_engine.cell(row=r, column=1, value=i).border = border_thin
        
        # --- CURRENT POSTURE ---
        ws_engine.cell(row=r, column=2, value=f"=MAX(0,{get_stable_pert(c_map['CF'])})").number_format = "0.0"
        ws_engine.cell(row=r, column=3, value=f"=MAX(0,MIN(1,{get_stable_pert(c_map['PoA'])}))").number_format = "0.0%"
        
        f_c_tef = f"=IF(OR('2. Data Entry (Current)'!$E$4=4, '2. Data Entry (Current)'!$E$4=5), B{r}*C{r}, IF('2. Data Entry (Current)'!$E$4=1, 0, MAX(0, {get_stable_pert(c_map['TEF'])})))"
        ws_engine.cell(row=r, column=4, value=f_c_tef).number_format = "0.0"
        
        ws_engine.cell(row=r, column=5, value=f"=MAX(0,MIN(1,{get_stable_pert(c_map['TCap'])}))").number_format = "0.0%"
        ws_engine.cell(row=r, column=6, value=f"=MAX(0,MIN(1,{get_stable_pert(c_map['RS'])}))").number_format = "0.0%"
        
        f_c_vuln = f"=IF(OR('2. Data Entry (Current)'!$E$4=3, '2. Data Entry (Current)'!$E$4=5), IF(E{r}>F{r},1,0), IF('2. Data Entry (Current)'!$E$4=1, 0, MAX(0, MIN(1, {get_stable_pert(c_map['Vuln'])}))))"
        ws_engine.cell(row=r, column=7, value=f_c_vuln).number_format = "0.0%"
        
        f_c_lef = f"=IF('2. Data Entry (Current)'!$E$4=1, MAX(0, {get_stable_pert(c_map['LEF'])}), D{r}*G{r})"
        ws_engine.cell(row=r, column=8, value=f_c_lef).number_format = "0.0"
        
        ws_engine.cell(row=r, column=9, value=f"=MAX(0,ROUND(H{r},0))").number_format = "#,##0"
        
        for ev in range(5):
            ws_engine.cell(row=r, column=10 + ev, value=get_prim_loss_formula(False)).number_format = "$#,##0"
        
        for ev in range(5):
            sec_calc = (
                f"={get_sec_loss_formula(False, c_map['S_Resp_F'], c_map['S_Resp_M'])}+"
                f"{get_sec_loss_formula(False, c_map['S_Comp_F'], c_map['S_Comp_M'])}+"
                f"{get_sec_loss_formula(False, c_map['S_Fine_F'], c_map['S_Fine_M'])}+"
                f"{get_sec_loss_formula(False, c_map['S_Repu_F'], c_map['S_Repu_M'])}"
            )
            ws_engine.cell(row=r, column=15 + ev, value=sec_calc).number_format = "$#,##0"
            
        ws_engine.cell(row=r, column=20, value=f"=IF(I{r}>=1,J{r},0)+IF(I{r}>=2,K{r},0)+IF(I{r}>=3,L{r},0)+IF(I{r}>=4,M{r},0)+IF(I{r}>=5,N{r},0)").number_format = "$#,##0"
        ws_engine.cell(row=r, column=21, value=f"=IF(I{r}>=1,O{r},0)+IF(I{r}>=2,P{r},0)+IF(I{r}>=3,Q{r},0)+IF(I{r}>=4,R{r},0)+IF(I{r}>=5,S{r},0)").number_format = "$#,##0"
        ws_engine.cell(row=r, column=22, value=f"=T{r}+U{r}").number_format = "$#,##0"

        # --- PROPOSED POSTURE ---
        ws_engine.cell(row=r, column=23, value=f"=MAX(0,{get_stable_pert(c_map['CF'], is_proposed=True)})").number_format = "0.0"
        ws_engine.cell(row=r, column=24, value=f"=MAX(0,MIN(1,{get_stable_pert(c_map['PoA'], is_proposed=True)}))").number_format = "0.0%"
        
        f_p_tef = f"=IF(OR('3. Data Entry (Proposed)'!$E$4=4, '3. Data Entry (Proposed)'!$E$4=5), W{r}*X{r}, IF('3. Data Entry (Proposed)'!$E$4=1, 0, MAX(0, {get_stable_pert(c_map['TEF'], is_proposed=True)})))"
        ws_engine.cell(row=r, column=25, value=f_p_tef).number_format = "0.0"
        
        ws_engine.cell(row=r, column=26, value=f"=MAX(0,MIN(1,{get_stable_pert(c_map['TCap'], is_proposed=True)}))").number_format = "0.0%"
        ws_engine.cell(row=r, column=27, value=f"=MAX(0,MIN(1,{get_stable_pert(c_map['RS'], is_proposed=True)}))").number_format = "0.0%"
        
        f_p_vuln = f"=IF(OR('3. Data Entry (Proposed)'!$E$4=3, '3. Data Entry (Proposed)'!$E$4=5), IF(Z{r}>AA{r},1,0), IF('3. Data Entry (Proposed)'!$E$4=1, 0, MAX(0, MIN(1, {get_stable_pert(c_map['Vuln'], is_proposed=True)}))))"
        ws_engine.cell(row=r, column=28, value=f_p_vuln).number_format = "0.0%"
        
        f_p_lef = f"=IF('3. Data Entry (Proposed)'!$E$4=1, MAX(0, {get_stable_pert(c_map['LEF'], is_proposed=True)}), Y{r}*AB{r})"
        ws_engine.cell(row=r, column=29, value=f_p_lef).number_format = "0.0"
        
        ws_engine.cell(row=r, column=30, value=f"=MAX(0,ROUND(AC{r},0))").number_format = "#,##0"
        
        for ev in range(5):
            ws_engine.cell(row=r, column=31 + ev, value=get_prim_loss_formula(True)).number_format = "$#,##0"
        
        for ev in range(5):
            sec_calc_p = (
                f"={get_sec_loss_formula(True, c_map['S_Resp_F'], c_map['S_Resp_M'])}+"
                f"{get_sec_loss_formula(True, c_map['S_Comp_F'], c_map['S_Comp_M'])}+"
                f"{get_sec_loss_formula(True, c_map['S_Fine_F'], c_map['S_Fine_M'])}+"
                f"{get_sec_loss_formula(True, c_map['S_Repu_F'], c_map['S_Repu_M'])}"
            )
            ws_engine.cell(row=r, column=36 + ev, value=sec_calc_p).number_format = "$#,##0"
            
        ws_engine.cell(row=r, column=41, value=f"=IF(AD{r}>=1,AE{r},0)+IF(AD{r}>=2,AF{r},0)+IF(AD{r}>=3,AG{r},0)+IF(AD{r}>=4,AH{r},0)+IF(AD{r}>=5,AI{r},0)").number_format = "$#,##0"
        ws_engine.cell(row=r, column=42, value=f"=IF(AD{r}>=1,AJ{r},0)+IF(AD{r}>=2,AK{r},0)+IF(AD{r}>=3,AL{r},0)+IF(AD{r}>=4,AM{r},0)+IF(AD{r}>=5,AN{r},0)").number_format = "$#,##0"
        ws_engine.cell(row=r, column=43, value=f"=AO{r}+AP{r}").number_format = "$#,##0"
        
        for col in range(1, 44):
            ws_engine.cell(row=r, column=col).border = border_thin

    for col_num in range(58, 71):
        ws_engine.column_dimensions[get_column_letter(col_num)].hidden = True

    # Auto-adjust column layouts
    for sheet in wb.worksheets:
        for col in sheet.columns:
            col_letter = get_column_letter(col[0].column)
            if sheet.title == ws_engine.title and col[0].column <= 57:
                sheet.column_dimensions[col_letter].width = 33
                continue
            max_len = max(len(str(cell.value or '')) for cell in col if not str(cell.value or '').startswith('='))
            sheet.column_dimensions[col_letter].width = max(max_len + 4, 15)

    ws_current.column_dimensions['A'].width = 65
    ws_current.column_dimensions['F'].width = 45
    ws_current.column_dimensions['G'].width = 35
    ws_proposed.column_dimensions['A'].width = 65
    ws_proposed.column_dimensions['F'].width = 45
    ws_proposed.column_dimensions['G'].width = 35

    ws_dash.column_dimensions['A'].width = 45
    ws_dash.column_dimensions['B'].width = 24
    ws_dash.column_dimensions['C'].width = 24
    ws_dash.column_dimensions['D'].width = 22
    ws_dash.column_dimensions['E'].width = 28

    output_filename = "Open_FAIR_Risk_Analysis_Tool_v15.xlsx"
    wb.save(output_filename)
    print(f"Generated clean validation workbook: {output_filename}")

if __name__ == "__main__":
    create_v15_workbook()